# Récupération et segmentation — BSB Munich

Télécharge les pages d'un ou plusieurs ouvrages depuis la bibliothèque numérique de Munich (BSB) via IIIF, puis segmente les illustrations avec YOLO.

**Sources** : `https://api.digitale-sammlungen.de/iiif/presentation/v2/{id}/manifest`

## 1. Imports et configuration

In [ ]:
import os, sys

# Chemins — adapter selon l'emplacement du notebook
RACINE = os.path.abspath("../..")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

from gallica_utils import telecharger_pages_iiif, segmenter_corpus, charger_yolo, liberer_yolo

DOSSIER_SOURCES = os.path.join(RACINE, "data", "sources")
DOSSIER_SEG     = os.path.join(RACINE, "data", "segmentees")

print(f"Racine     : {RACINE}")
print(f"Sources    : {DOSSIER_SOURCES}")
print(f"Segmentées : {DOSSIER_SEG}")

## 2. Sources à récupérer

Renseigner l'identifiant BSB et le nom du dossier de destination (convention `{technique}_{graveur}_{editeur}_{ville}{annee}`).

In [ ]:
SOURCES_BSB = {
    # "bsb_id" : "nom_dossier",
    "bsb00008186" : "cuivre_exemple_editeur_ville1610",
}

## 3. Vérification — nombre de pages disponibles

In [ ]:
import requests

for bsb_id, nom in SOURCES_BSB.items():
    url = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
    r   = requests.get(url, timeout=15)
    if r.status_code == 200:
        pages = len(r.json()["sequences"][0]["canvases"])
        print(f"OK  {nom:45s} : {pages} pages")
    else:
        print(f"ERR {nom:45s} : HTTP {r.status_code}")

## 4. Charger le modèle YOLO

In [ ]:
modele_yolo = charger_yolo()
print("YOLO chargé")

## 5. Téléchargement et segmentation

In [ ]:
for bsb_id, nom in SOURCES_BSB.items():
    print(f"\n{'='*60}\n{nom}\n{'='*60}")

    dossier_source = os.path.join(DOSSIER_SOURCES, nom)

    # Charger les pages déjà téléchargées si présentes, sinon télécharger
    if os.path.exists(dossier_source) and len(os.listdir(dossier_source)) > 0:
        pages = sorted([
            os.path.join(dossier_source, f)
            for f in os.listdir(dossier_source) if f.endswith(".jpg")
        ])
        print(f"{len(pages)} pages déjà disponibles")
    else:
        pages = telecharger_pages_iiif(
            f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
            dossier_source,
            prefixe=nom
        )

    # Segmenter les illustrations
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, nom),
        modele_yolo,
        conf_thres=0.25
    )

## 6. Libérer le modèle YOLO

In [ ]:
liberer_yolo(modele_yolo)
print("YOLO libéré")

## 7. Récapitulatif

In [ ]:
print("Récapitulatif :\n")
for bsb_id, nom in SOURCES_BSB.items():
    chemin = os.path.join(DOSSIER_SEG, nom)
    if os.path.exists(chemin):
        n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
        print(f"  {nom:45s} : {n} illustrations")
    else:
        print(f"  {nom:45s} : (non traité)")

In [16]:
import requests, re

def recuperer_tous_les_ids(query, page_size=100, filtres=None, max_pages=50):
    """
    Récupère tous les IDs de documents d'une recherche MDZ via l'API.
    Retourne une liste de dicts {id, titre, lieu, date}.
    """
    url_api = "https://www.digitale-sammlungen.de/api/search"
    tous = []
    page = 0

    while page < max_pages:
        params = {
            "query": query,
            "handler": "simple-metadata",
            "sortField": "date",
            "sortOrder": "asc",
            "page": page,
            "pageSize": page_size,
        }
        # Ajouter les filtres (liste de chaînes "champ:valeur")
        if filtres:
            params["filter"] = filtres

        r = requests.get(url_api, params=params, timeout=20)
        if r.status_code != 200:
            print(f"  Arrêt — HTTP {r.status_code} à la page {page}")
            break

        data = r.json()
        docs = data.get("docs", [])
        if not docs:
            break

        for d in docs:
            tous.append({
                "id"    : d.get("id"),
                "titre" : re.sub(r"<[^>]+>", "", d.get("title", "")),  # retirer les balises <em>
                "lieu"  : ", ".join(d.get("publicationPlaces", [])),
                "date"  : d.get("publicationDate", ""),
            })

        total = data.get("numTotal", 0)
        print(f"  page {page} — {len(tous)}/{total} documents récupérés", end="\r")

        if len(tous) >= total:
            break
        page += 1

    print(f"\n✓ {len(tous)} documents récupérés")
    return tous


# ── Bibles illustrées 1551-1750 ───────
docs = recuperer_tous_les_ids(
    query="(biblia tafeln)",
    page_size=100,
    filtres=[
        'type_manufact:-"handmade"',
        'date_facet:[1551-01-01 TO 1750-06-16]',
    ],
)

# Aperçu
print(f"\nAperçu des premiers documents :\n")
for d in docs[:10]:
    print(f"  {d['id']} — {d['date']:12s} — {d['lieu']:20s} — {d['titre'][:50]}")

  page 19 — 2000/1944 documents récupérés
✓ 2000 documents récupérés

Aperçu des premiers documents :

  bsb11283985 — 1551         — Parisiis             — Biblia sancta : vetus testamentum novumque unius e
  bsb11203230 — 1551         — Lyon                 — La Bible en Francoys : qui est toute la saincte es
  bsb11095955 — 1551         — Franc[ofurti]        — Biblia Veteris Testamenti et Historiae, Artificios
  bsb10205700 — 1551         — Friburg im Breyßgaw  — Das new Testament
  bsb11095956 — [1551]       — Franc[ofurti]        — Novi Testamenti, Iesv Christi Historia Effigiata :
  bsb00085158 — 1551         — Franc[ofurti]        — Apocalypsis S. Joannis : = Die Offenbarung S. Joha
  bsb00085156 — 1551         — Franc.               — Biblia Veteris Testamenti & Histori[a]e, artificio
  bsb00085342 — [1551]       — Franc[ofurti]        — Novi Testamenti, Iesv Christi Historia Effigiata :
  bsb00085447 — 1551         — Wittemberg           — Biblia Das ist: Die gantze heilige S

In [20]:
# ── Liste des documents À PARTIR du 59e (les 58 premiers déjà vus) ──
chemin_txt = "resultats/bibles_mdz_a_choisir.txt"

docs_restants = docs[58:]   # on saute les 58 premiers

# Largeurs de colonnes pour un alignement propre
LARG_ID   = 14
LARG_DATE = 12
LARG_LIEU = 22

with open(chemin_txt, "w", encoding="utf-8") as f:
    f.write("╔" + "═" * 78 + "╗\n")
    f.write("║" + "LISTE DES BIBLES ILLUSTRÉES — MDZ".center(78) + "║\n")
    f.write("║" + "(à partir du 59e document)".center(78) + "║\n")
    f.write("╚" + "═" * 78 + "╝\n\n")
    f.write("Consigne : gardez uniquement les lignes des documents qui vous intéressent,\n")
    f.write("supprimez les autres, puis renvoyez-moi ce fichier.\n\n")
    f.write("─" * 80 + "\n")
    f.write(f"{'IDENTIFIANT':<{LARG_ID}}{'DATE':<{LARG_DATE}}{'LIEU':<{LARG_LIEU}}TITRE\n")
    f.write("─" * 80 + "\n\n")

    for d in docs_restants:
        ident = (d["id"] or "")[:LARG_ID-1]
        date  = (d["date"] or "")[:LARG_DATE-1]
        lieu  = (d["lieu"] or "")[:LARG_LIEU-1]
        titre = d["titre"] or ""
        f.write(f"{ident:<{LARG_ID}}{date:<{LARG_DATE}}{lieu:<{LARG_LIEU}}{titre}\n")

print(f"✓ Fichier généré : {chemin_txt}")
print(f"  {len(docs_restants)} documents listés (du 59e à la fin)")

✓ Fichier généré : resultats/bibles_mdz_a_choisir.txt
  1942 documents listés (du 59e à la fin)


In [21]:
# Dédoublonner par id et garder le bon total
vus = set()
docs_uniques = []
for d in docs:
    if d["id"] not in vus:
        vus.add(d["id"])
        docs_uniques.append(d)

docs = docs_uniques
print(f"{len(docs)} documents uniques")

100 documents uniques


In [27]:
import requests

for taille in [500, 1000, 2000]:
    params = {
        "query": "(biblia tafeln)", "handler": "simple-metadata",
        "sortField": "date", "sortOrder": "asc", "pageSize": taille,
        "filter": ['type_manufact:-"handmade"', 'date_facet:[1551-01-01 TO 1750-06-16]'],
    }
    r = requests.get("https://www.digitale-sammlungen.de/api/search", params=params, timeout=30)
    n = len(r.json()["docs"]) if r.status_code == 200 else 0
    print(f"  pageSize={taille:5d} → {n} documents reçus")

  pageSize=  500 → 250 documents reçus
  pageSize= 1000 → 250 documents reçus
  pageSize= 2000 → 250 documents reçus


In [17]:
import pandas as pd

# docs est déjà en mémoire
df_bibles = pd.DataFrame(docs)

# Ajouter le lien MDZ pour chaque document
df_bibles["lien_mdz"] = "https://www.digitale-sammlungen.de/en/view/" + df_bibles["id"]

print(f"{len(df_bibles)} documents")
print(df_bibles.head())

# Sauvegarder en CSV
chemin = "resultats/csv/bibles_mdz_recherche.csv"
df_bibles.to_csv(chemin, index=False, encoding="utf-8-sig")
print(f"\n✓ Sauvegardé : {chemin}")

2000 documents
            id                                              titre  \
0  bsb11283985  Biblia sancta : vetus testamentum novumque uni...   
1  bsb11203230  La Bible en Francoys : qui est toute la sainct...   
2  bsb11095955  Biblia Veteris Testamenti et Historiae, Artifi...   
3  bsb10205700                                  Das new Testament   
4  bsb11095956  Novi Testamenti, Iesv Christi Historia Effigia...   

                  lieu    date  \
0             Parisiis    1551   
1                 Lyon    1551   
2        Franc[ofurti]    1551   
3  Friburg im Breyßgaw    1551   
4        Franc[ofurti]  [1551]   

                                            lien_mdz  
0  https://www.digitale-sammlungen.de/en/view/bsb...  
1  https://www.digitale-sammlungen.de/en/view/bsb...  
2  https://www.digitale-sammlungen.de/en/view/bsb...  
3  https://www.digitale-sammlungen.de/en/view/bsb...  
4  https://www.digitale-sammlungen.de/en/view/bsb...  

✓ Sauvegardé : resultats/csv/bible

In [12]:
import os, sys, requests
from PIL import Image
from io import BytesIO

RACINE = os.path.abspath(".")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

from gallica_utils import segmenter_page, charger_yolo, liberer_yolo

DOSSIER_SOURCES = os.path.join(RACINE, "data", "sources")
DOSSIER_SEG     = os.path.join(RACINE, "data", "segmentees")
print("✓ Imports OK")


def recuperer_n_illustrations(bsb_id, nom_dossier, modele_yolo,
                               n_illustrations=8, conf_thres=0.25):
    """
    Télécharge et segmente les pages d'un document BSB une par une,
    jusqu'à obtenir n_illustrations. Le nom des fichiers encode
    l'id BSB et le numéro de page pour retrouver la source.
    """
    url      = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
    manifest = requests.get(url, timeout=20).json()
    canvases = manifest["sequences"][0]["canvases"]
    print(f"  Document : {len(canvases)} pages disponibles")

    dossier_source = os.path.join(DOSSIER_SOURCES, nom_dossier)
    dossier_seg    = os.path.join(DOSSIER_SEG, nom_dossier)
    os.makedirs(dossier_source, exist_ok=True)
    os.makedirs(dossier_seg, exist_ok=True)

    total_illus = 0

    for i, canvas in enumerate(canvases):
        if total_illus >= n_illustrations:
            break

        page_num = i + 1
        img_url  = canvas["images"][0]["resource"]["@id"]

        # Nom encodant l'id BSB et le numéro de page
        chemin = os.path.join(dossier_source, f"{bsb_id}_page{page_num:03d}.jpg")
        if not os.path.exists(chemin):
            try:
                img = Image.open(BytesIO(requests.get(img_url, timeout=20).content)).convert("RGB")
                img.save(chemin)
            except Exception as e:
                print(f"\n  Erreur page {page_num} : {e}")
                continue

        # Le préfixe encode id BSB + page → les illustrations héritent de ce nom
        prefixe = f"{bsb_id}_page{page_num:03d}"
        nb = segmenter_page(chemin, prefixe, dossier_seg, modele_yolo, conf_thres)
        total_illus += nb
        print(f"  page {page_num} → {nb} illustration(s) | total : {total_illus}/{n_illustrations}", end="\r")

    print(f"\n✓ {total_illus} illustrations extraites en parcourant {i+1} pages → {nom_dossier}")
    if total_illus < n_illustrations:
        print(f"  ⚠️  Le document ne contient que {total_illus} illustration(s) au total.")
    return total_illus


# ── Document à traiter ──────────────────────────────────────
BSB_ID = "bsb11283985"
NOM    = "biblia_sancta_bsb11283985"

modele_yolo = charger_yolo()
print(f"\n{'='*55}\n{NOM}\n{'='*55}")
recuperer_n_illustrations(BSB_ID, NOM, modele_yolo, n_illustrations=10)
liberer_yolo(modele_yolo)
print("\n✓ Terminé")

✓ Imports OK
✓ YOLO chargé — classes : {0: 'illustration'}

biblia_sancta_bsb11283985
  Document : 1500 pages disponibles
  page 31 → 1 illustration(s) | total : 10/10
✓ 10 illustrations extraites en parcourant 32 pages → biblia_sancta_bsb11283985
✓ Mémoire GPU libérée

✓ Terminé


In [8]:
import requests

bsb_id   = "bsb11283985"
url      = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
manifest = requests.get(url, timeout=20).json()
nb_pages = len(manifest["sequences"][0]["canvases"])
print(f"{nb_pages} pages")

1500 pages


In [34]:
import requests, re

def recuperer_pages(query, filtres=None, page_size=100, page_debut=1, page_fin=20):
    """Récupère les documents des pages page_debut à page_fin (incluses).
    Affiche le détail de chaque page sur une ligne distincte."""
    url_api = "https://www.digitale-sammlungen.de/api/search"
    tous = []

    for sp in range(page_debut, page_fin + 1):
        params = {
            "query": query, "handler": "simple-metadata", "ocrContext": 1,
            "sortField": "date", "sortOrder": "asc",
            "startPage": sp, "pageSize": page_size,
        }
        if filtres:
            params["filter"] = filtres

        r = requests.get(url_api, params=params, timeout=30)
        if r.status_code != 200:
            print(f"  startPage {sp:2d} → HTTP {r.status_code}")
            continue

        docs = r.json().get("docs", [])
        for d in docs:
            tous.append({
                "id"    : d.get("id"),
                "titre" : re.sub(r"<[^>]+>", "", d.get("title", "")),
                "lieu"  : ", ".join(d.get("publicationPlaces", [])),
                "date"  : d.get("publicationDate", ""),
            })
        # Une ligne par page : nb de docs de la page + cumul
        print(f"  startPage {sp:2d} → {len(docs):3d} docs sur cette page | cumul : {len(tous)}")

    print(f"\n✓ {len(tous)} documents récupérés (pages {page_debut} à {page_fin})")
    return tous


docs_final = recuperer_pages(
    query="(biblia tafeln)",
    filtres=['type_manufact:-"handmade"', 'date_facet:[1551-01-01 TO 1750-06-16]'],
    page_debut=0, page_fin=20,
)

  startPage  0 → 100 docs sur cette page | cumul : 100
  startPage  1 → 100 docs sur cette page | cumul : 200
  startPage  2 → 100 docs sur cette page | cumul : 300
  startPage  3 → 100 docs sur cette page | cumul : 400
  startPage  4 → 100 docs sur cette page | cumul : 500
  startPage  5 → 100 docs sur cette page | cumul : 600
  startPage  6 → 100 docs sur cette page | cumul : 700
  startPage  7 → 100 docs sur cette page | cumul : 800
  startPage  8 → 100 docs sur cette page | cumul : 900
  startPage  9 → 100 docs sur cette page | cumul : 1000
  startPage 10 → 100 docs sur cette page | cumul : 1100
  startPage 11 → 100 docs sur cette page | cumul : 1200
  startPage 12 → 100 docs sur cette page | cumul : 1300
  startPage 13 → 100 docs sur cette page | cumul : 1400
  startPage 14 → 100 docs sur cette page | cumul : 1500
  startPage 15 → 100 docs sur cette page | cumul : 1600
  startPage 16 → 100 docs sur cette page | cumul : 1700
  startPage 17 → 100 docs sur cette page | cumul : 1800
 

In [35]:
# ── Tableau HTML de sélection — 1944 Bibles, suppression + sauvegarde + export ──
import os, json

chemin_html = "resultats/selection_bibles_mdz.html"

docs_json = json.dumps([
    {"id": d["id"], "date": d["date"] or "", "lieu": d["lieu"] or "", "titre": d["titre"] or ""}
    for d in docs_final
], ensure_ascii=False)

html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Sélection des Bibles illustrées — MDZ</title>
<style>
  body {{ font-family: Georgia, serif; margin: 30px; color: #2c2c2c; background: #faf8f5; }}
  h1 {{ font-size: 22px; color: #5a3e2b; border-bottom: 2px solid #c89b6a; padding-bottom: 10px; }}
  .sous-titre {{ color: #777; font-size: 14px; margin-bottom: 20px; }}
  .barre {{ position: sticky; top: 0; background: #faf8f5; padding: 15px 0; z-index: 10;
            border-bottom: 1px solid #ddd; margin-bottom: 10px; }}
  button {{ background: #5a3e2b; color: white; border: none; padding: 10px 20px; font-size: 14px;
            border-radius: 4px; cursor: pointer; font-family: Georgia, serif; margin-right: 10px; }}
  button:hover {{ background: #7a5638; }}
  .reset {{ background: #999; }}
  #compteur {{ margin-left: 5px; color: #777; font-size: 13px; }}
  table {{ border-collapse: collapse; width: 100%; background: white; }}
  th {{ background: #5a3e2b; color: white; padding: 10px; font-size: 13px; text-align: left;
        position: sticky; top: 70px; }}
  td {{ padding: 8px 10px; border-bottom: 1px solid #eee; font-size: 13px; vertical-align: middle; }}
  tr:hover {{ background: #f5efe6; }}
  .num {{ color: #aaa; font-size: 12px; width: 45px; }}
  .ident {{ font-family: monospace; font-size: 12px; color: #555; }}
  .titre {{ max-width: 430px; }}
  .suppr {{ background: #c0392b; color: white; border: none; border-radius: 50%;
            width: 26px; height: 26px; cursor: pointer; font-size: 14px; line-height: 1; }}
  .suppr:hover {{ background: #e74c3c; }}
</style>
</head>
<body>
  <h1>Sélection des Bibles illustrées (MDZ)</h1>
  <p class="sous-titre">
    Cliquez sur la croix rouge pour retirer les documents qui ne vous intéressent pas.
    Votre sélection est <b>sauvegardée automatiquement</b> — vous pouvez fermer et reprendre.
    Quand vous avez terminé, cliquez sur « Télécharger ma sélection » et envoyez-moi le fichier.
  </p>
  <div class="barre">
    <button onclick="telecharger()">⬇ Télécharger ma sélection</button>
    <button class="reset" onclick="reinitialiser()">Tout réafficher</button>
    <span id="compteur"></span>
  </div>
  <table>
    <thead>
      <tr><th class="num">#</th><th>Identifiant</th><th>Date</th><th>Lieu</th><th>Titre</th><th></th></tr>
    </thead>
    <tbody id="corps"></tbody>
  </table>

<script>
  const DOCS = {docs_json};
  const STORAGE_KEY = 'selection_bibles_mdz';

  function charger() {{
    const d = localStorage.getItem(STORAGE_KEY);
    return d ? JSON.parse(d) : [];
  }}
  function sauver(supprimes) {{
    localStorage.setItem(STORAGE_KEY, JSON.stringify(supprimes));
  }}

  function rendre() {{
    const supprimes = charger();
    const corps = document.getElementById('corps');
    corps.innerHTML = '';
    let n = 0;
    DOCS.forEach(doc => {{
      if (supprimes.includes(doc.id)) return;
      n++;
      const tr = document.createElement('tr');
      tr.innerHTML =
        '<td class="num">' + n + '</td>' +
        '<td class="ident">' + doc.id + '</td>' +
        '<td>' + doc.date + '</td>' +
        '<td>' + doc.lieu + '</td>' +
        '<td class="titre">' + doc.titre + '</td>' +
        '<td><button class="suppr" title="Retirer">✕</button></td>';
      tr.querySelector('.suppr').addEventListener('click', () => {{
        const s = charger();
        s.push(doc.id);
        sauver(s);
        rendre();
      }});
      corps.appendChild(tr);
    }});
    document.getElementById('compteur').textContent =
      n + ' document(s) conservé(s) sur ' + DOCS.length;
  }}

  function telecharger() {{
    const supprimes = charger();
    const gardes = DOCS.filter(d => !supprimes.includes(d.id));
    let txt = 'SÉLECTION DES BIBLES ILLUSTRÉES — MDZ\\n';
    txt += gardes.length + ' documents conservés sur ' + DOCS.length + '\\n';
    txt += '='.repeat(70) + '\\n\\n';
    gardes.forEach((d, i) => {{
      txt += (i+1) + '. ' + d.id + ' | ' + d.date + ' | ' + d.lieu + ' | ' + d.titre + '\\n';
    }});
    const blob = new Blob([txt], {{ type: 'text/plain;charset=utf-8;' }});
    const lien = document.createElement('a');
    lien.href = URL.createObjectURL(blob);
    lien.download = 'selection_bibles_celine.txt';
    lien.click();
  }}

  function reinitialiser() {{
    if (confirm('Réafficher tous les documents (annuler vos suppressions) ?')) {{
      localStorage.removeItem(STORAGE_KEY);
      rendre();
    }}
  }}

  rendre();
</script>
</body>
</html>"""

with open(chemin_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Tableau de sélection généré : {chemin_html}")
print(f"  {len(docs_final)} documents")

✓ Tableau de sélection généré : resultats/selection_bibles_mdz.html
  1944 documents


In [ ]:
import os, sys, re, requests, time
from PIL import Image
from io import BytesIO

RACINE = os.path.abspath("../../")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

from gallica_utils import segmenter_page, charger_yolo, liberer_yolo
print("✓ Imports OK")

DOSSIER_BIBLES_SEG = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")
os.makedirs(DOSSIER_BIBLES_SEG, exist_ok=True)

# ── Reprise : nb d'illustrations déjà extraites ─────────────
def deja_traite(bsb_id):
    dossier_seg = os.path.join(DOSSIER_BIBLES_SEG, bsb_id)
    if not os.path.isdir(dossier_seg):
        return None
    illus = [f for f in os.listdir(dossier_seg)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))
             and not f.startswith("_tmp_")]
    return len(illus)

# ── Téléchargement avec gestion du rate-limit 429 ───────────
def telecharger_avec_retry(url, max_essais=4, pause_base=2):
    for essai in range(max_essais):
        try:
            response = requests.get(url, timeout=20)
            if response.status_code == 429:
                attente = pause_base * (2 ** essai)
                print(f"      429 — pause {attente}s", end="\r")
                time.sleep(attente)
                continue
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert("RGB")
        except requests.exceptions.HTTPError:
            raise
        except Exception:
            time.sleep(pause_base)
    raise Exception("Échec après plusieurs essais (rate-limit persistant)")

# ── Extraction de N illustrations pour un document ──────────
def recuperer_n_illustrations(bsb_id, modele_yolo, n_illustrations=10, conf_thres=0.25):
    url = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
    try:
        manifest = requests.get(url, timeout=20).json()
    except Exception as e:
        print(f"  ✗ Manifest inaccessible : {e}")
        return 0
    canvases = manifest["sequences"][0]["canvases"]
    dossier_seg = os.path.join(DOSSIER_BIBLES_SEG, bsb_id)
    os.makedirs(dossier_seg, exist_ok=True)
    total_illus = 0
    for i, canvas in enumerate(canvases):
        if total_illus >= n_illustrations:
            break
        page_num = i + 1
        img_url  = canvas["images"][0]["resource"]["@id"]
        try:
            img = telecharger_avec_retry(img_url)
        except Exception as e:
            print(f"    page {page_num} ignorée : {e}")
            continue
        chemin_tmp = os.path.join(dossier_seg, f"_tmp_{bsb_id}_p{page_num:03d}.jpg")
        try:
            img.save(chemin_tmp)
            prefixe = f"{bsb_id}_page{page_num:03d}"
            nb = segmenter_page(chemin_tmp, prefixe, dossier_seg, modele_yolo, conf_thres)
            total_illus += nb
        except Exception as e:
            print(f"    erreur segmentation page {page_num} : {e}")
        finally:
            if os.path.exists(chemin_tmp):
                os.remove(chemin_tmp)
        time.sleep(0.5)
    return total_illus

# ── Lire la sélection ───────────────────────────────────────
chemin_selection = os.path.join(RACINE, "retours_celine", "selection_bibles_celine_1550-1750_Biblia.txt")
with open(chemin_selection, encoding="utf-8") as f:
    contenu = f.read()
ids = re.findall(r"bsb\d+", contenu)
print(f"{len(ids)} documents dans la sélection\n")

# ── Pipeline avec reprise ───────────────────────────────────
modele_yolo = charger_yolo()
resultats   = {}
nb_sautes   = 0

for k, bsb_id in enumerate(ids, 1):
    deja = deja_traite(bsb_id)
    if deja is not None:
        resultats[bsb_id] = deja
        nb_sautes += 1
        print(f"[{k}/{len(ids)}] {bsb_id} → déjà traité ({deja} illus), ignoré")
        continue
    print(f"[{k}/{len(ids)}] {bsb_id}", end=" → ")
    try:
        n = recuperer_n_illustrations(bsb_id, modele_yolo, n_illustrations=10)
        resultats[bsb_id] = n
        print(f"{n} illustrations")
    except Exception as e:
        resultats[bsb_id] = -1
        print(f"ERREUR : {e}")

liberer_yolo(modele_yolo)

# ── Récapitulatif ───────────────────────────────────────────
ok          = sum(1 for v in resultats.values() if v > 0)
vides       = sum(1 for v in resultats.values() if v == 0)
err         = sum(1 for v in resultats.values() if v == -1)
total_illus = sum(v for v in resultats.values() if v > 0)
print(f"\n{'='*55}")
print(f"✓ Terminé — {ok} docs avec illustrations, {vides} sans, {err} en erreur")
print(f"  {nb_sautes} documents déjà traités (sautés)")
print(f"  Total illustrations extraites : {total_illus}")

✓ Imports OK
398 documents dans la sélection



✓ YOLO chargé — classes : {0: 'illustration'}
[1/398] bsb11283985 → déjà traité (10 illus), ignoré
[2/398] bsb11203230 → déjà traité (10 illus), ignoré
[3/398] bsb11095955 → déjà traité (10 illus), ignoré
[4/398] bsb00085156 → déjà traité (10 illus), ignoré
[5/398] bsb00085447 → déjà traité (10 illus), ignoré
[6/398] bsb00096752 → déjà traité (10 illus), ignoré
[7/398] bsb10196335 → déjà traité (10 illus), ignoré
[8/398] bsb11203161 → déjà traité (10 illus), ignoré
[9/398] bsb11255749 → déjà traité (10 illus), ignoré
[10/398] bsb11283984 → déjà traité (10 illus), ignoré
[11/398] bsb11362144 → déjà traité (10 illus), ignoré
[12/398] bsb11224888 → déjà traité (10 illus), ignoré
[13/398] bsb11203163 → déjà traité (10 illus), ignoré
[14/398] bsb11925418 → déjà traité (10 illus), ignoré
[15/398] bsb10205772 → déjà traité (10 illus), ignoré
[16/398] bsb10196330 → déjà traité (10 illus), ignoré
[17/398] bsb11203165 → déjà traité (10 illus), ignoré
[18/398] bsb10141273 → déjà traité (10 illus)

In [ ]:
https://www.digitale-sammlungen.de/en/view/ ?page= 